In [3]:
!sudo apt-get update -y
!sudo apt-get install -y xvfb

!pip install -U pip

# latest replacements for your old stack (no version pins)
!pip install -U gymnasium pyvirtualdisplay pytorch-lightning

import warnings
warnings.filterwarnings("ignore")

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install "gymnasium[mujoco]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 99.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mujoco]


In [6]:
from pyvirtualdisplay import Display
Display(visible=False, size=(1400, 900)).start()

In [7]:
import copy
import torch
import random
import gymnasium as gym
import matplotlib
import functools
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
import io, base64
import imageio.v2 as imageio
from IPython.display import HTML

import torch.nn.functional as F

from collections import deque, namedtuple
from IPython.display import HTML
from base64 import b64encode

from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data.dataset import IterableDataset
from torch.optim import AdamW

from torch.distributions import Normal

from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from gymnasium.wrappers import RecordVideo, RecordEpisodeStatistics
from gymnasium.wrappers.vector import NormalizeObservation, NormalizeReward
from gymnasium.wrappers import NormalizeObservation as NO

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
num_gpus = torch.cuda.device_count()

v = torch.ones(1, device=device)

In [8]:
@torch.no_grad()
def test_agent(env, episode_length, policy, episodes=10):
    device = next(policy.parameters()).device

    ep_returns = []
    for _ in range(episodes):
        obs, _ = env.reset()
        ep_ret = 0.0

        for _ in range(episode_length):
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)

            loc, scale = policy(obs_t)
            sample = torch.normal(loc, scale)
            action = torch.tanh(sample)

            action_np = action.detach().cpu().numpy()
            obs, reward, terminated, truncated, _ = env.step(action_np)

            ep_ret += float(reward)
            if terminated or truncated:
                break

        ep_returns.append(ep_ret)

    return sum(ep_returns) / episodes


@torch.no_grad()
def create_video(env, episode_length, policy=None, fps=30):
    frames = []
    obs, _ = env.reset()

    device = None
    if policy is not None:
        device = next(policy.parameters()).device

    for _ in range(episode_length):
        frames.append(env.render())

        if policy is None:
            action_np = env.action_space.sample()
        else:
            obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
            loc, scale = policy(obs_t)
            sample = torch.normal(loc, scale)
            action = torch.tanh(sample)
            action_np = action.detach().cpu().numpy()

        obs, _, terminated, truncated, _ = env.step(action_np)
        if terminated or truncated:
            break

    buf = io.BytesIO()
    imageio.mimwrite(buf, frames, format="mp4", fps=fps)
    mp4 = base64.b64encode(buf.getvalue()).decode("ascii")
    return HTML(f"""
    <video width="600" controls>
      <source src="data:video/mp4;base64,{mp4}" type="video/mp4">
    </video>
    """)

In [9]:
checkpoint_callback = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/RL_Competition/ppo_checkpoints",
    filename="ppo-{epoch:03d}-{Average_return:.2f}",
    save_top_k=3,
    monitor="Average_return",
    mode="max",
    save_last=True,
    every_n_epochs=1
)

In [10]:
def create_env(name, num_envs=10):
    env = gym.make_vec(name, num_envs=num_envs, render_mode="rgb_array")
    env = NormalizeObservation(env)
    env = NormalizeReward(env, gamma=0.99)
    return env

In [11]:
class GradientPolicy(nn.Module):
    def __init__(self, input_dimensions, output_dimensions, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(input_dimensions, hidden_size)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.ln3 = nn.LayerNorm(hidden_size)

        self.mean = nn.Linear(hidden_size, output_dimensions)
        self.log_std = nn.Parameter(torch.zeros(output_dimensions))


        nn.init.orthogonal_(self.fc1.weight, gain=np.sqrt(2))
        nn.init.orthogonal_(self.fc2.weight, gain=np.sqrt(2))
        nn.init.orthogonal_(self.fc3.weight, gain=np.sqrt(2))
        nn.init.orthogonal_(self.mean.weight, gain=0.01)

    def forward(self, x):
        x = F.tanh(self.ln1(self.fc1(x)))
        x = F.tanh(self.ln2(self.fc2(x)))
        x = F.tanh(self.ln3(self.fc3(x)))

        mean = self.mean(x)
        std = torch.exp(self.log_std.clamp(-20, 2))

        return mean, std

In [12]:
class ValueNetwork(nn.Module):
    def __init__(self, input_dimensions, output_dimensions, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(input_dimensions, hidden_size)
        self.ln1 = nn.LayerNorm(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.ln3 = nn.LayerNorm(hidden_size)
        self.fc4 = nn.Linear(hidden_size, output_dimensions)


        nn.init.orthogonal_(self.fc1.weight, gain=np.sqrt(2))
        nn.init.orthogonal_(self.fc2.weight, gain=np.sqrt(2))
        nn.init.orthogonal_(self.fc3.weight, gain=np.sqrt(2))
        nn.init.orthogonal_(self.fc4.weight, gain=1.0)

    def forward(self, x):
        x = F.tanh(self.ln1(self.fc1(x)))
        x = F.tanh(self.ln2(self.fc2(x)))
        x = F.tanh(self.ln3(self.fc3(x)))
        x = self.fc4(x)
        return x

In [13]:
class RLDataset(IterableDataset):
    def __init__(self, env, policy, value_network, steps_per_rollout, discount_factor, gae_lambda, rollout_multiple):
        self.env = env
        self.policy = policy
        self.value_network = value_network
        self.steps_per_rollout = steps_per_rollout
        self.discount_factor = discount_factor
        self.gae_lambda = gae_lambda
        self.rollout_multiple = rollout_multiple
        self.state, _ = env.reset()
        self.state = torch.tensor(self.state).float()

    def __iter__(self):
        transitions = []
        policy_device = next(self.policy.parameters()).device

        for sample in range(self.steps_per_rollout):
            state_dev = self.state.to(policy_device)
            mean_dev, std_dev = self.policy(state_dev)


            dist = Normal(mean_dev, std_dev)
            action = dist.sample()
            log_prob = dist.log_prob(action).sum(dim=-1, keepdim=True)


            action_clipped = action.clamp(-1.0, 1.0)

            next_state, reward, terminated, truncated, _ = self.env.step(action_clipped.detach().cpu().numpy())
            next_state = torch.tensor(next_state).float()
            reward = torch.tensor(reward).unsqueeze(1).float()
            terminated = torch.tensor(terminated, dtype=torch.bool).unsqueeze(1)
            truncated = torch.tensor(truncated, dtype=torch.bool).unsqueeze(1)
            done = terminated | truncated


            with torch.no_grad():
                value = self.value_network(state_dev).detach().cpu()

            transitions.append((
                self.state,
                action.detach().cpu(),
                log_prob.detach().cpu(),
                reward,
                done,
                next_state,
                value
            ))
            self.state = next_state


        number_of_samples = self.env.num_envs * self.steps_per_rollout
        state_b, action_b, log_prob_b, reward_b, done_b, next_state_b, value_b = map(torch.stack, zip(*transitions))


        with torch.no_grad():
            final_value = self.value_network(next_state_b[-1].to(policy_device)).detach().cpu()


        advantages = torch.zeros_like(reward_b)
        last_gae = 0

        for t in reversed(range(self.steps_per_rollout)):
            if t == self.steps_per_rollout - 1:
                next_value = final_value
            else:
                next_value = value_b[t + 1]

            next_non_terminal = ~done_b[t]
            delta = reward_b[t] + self.discount_factor * next_value * next_non_terminal - value_b[t]
            last_gae = delta + self.discount_factor * self.gae_lambda * next_non_terminal * last_gae
            advantages[t] = last_gae

        returns = advantages + value_b

        # Flatten all tensors
        state_b = state_b.view(number_of_samples, -1)
        action_b = action_b.view(number_of_samples, -1)
        log_prob_b = log_prob_b.view(number_of_samples, -1)
        reward_b = reward_b.view(number_of_samples, -1)
        next_state_b = next_state_b.view(number_of_samples, -1)
        done_b = done_b.view(number_of_samples, -1)
        advantages_b = advantages.view(number_of_samples, -1)
        returns_b = returns.view(number_of_samples, -1)


        advantages_b = (advantages_b - advantages_b.mean()) / (advantages_b.std() + 1e-8)   # Normalize advantages

        for repeat in range(self.rollout_multiple):
            indexes = list(range(number_of_samples))
            random.shuffle(indexes)

            for index in indexes:
                yield (state_b[index], action_b[index], log_prob_b[index],
                       advantages_b[index], returns_b[index])

In [14]:
class PPO(LightningModule):
    def __init__(
        self,
        name,
        batch_size=2048,
        number=1000,
        steps_per_rollout=2048,
        rollout_multiple=10,
        discount_factor=0.99,
        gae_lambda=0.95,
        num_envs=16,
        policy_lr=3e-4,
        value_lr=1e-3,
        epsilon=0.2,
        entropy_weight=0.01,
        max_grad_norm=0.5,
        optim=AdamW,
        hidden=256
    ):
        super().__init__()
        self.automatic_optimization = False
        self.env = create_env(name, num_envs)
        self.policy = GradientPolicy(
            self.env.observation_space.shape[-1],
            self.env.action_space.shape[-1],
            hidden
        )
        self.value_network = ValueNetwork(
            self.env.observation_space.shape[-1],
            1,
            hidden
        )
        self.dataset = RLDataset(
            self.env,
            self.policy,
            self.value_network,
            steps_per_rollout,
            discount_factor,
            gae_lambda,
            rollout_multiple
        )

        test_env = gym.make(name, render_mode="rgb_array")
        self.test_env = NO(test_env)
        self.test_env.obs_rms = self.env.env.obs_rms

        self.save_hyperparameters()
        self.videos = []

    def configure_optimizers(self):
        policy_optimizer = self.hparams.optim(
            self.policy.parameters(),
            lr=self.hparams.policy_lr,
            eps=1e-5
        )
        value_optimizer = self.hparams.optim(
            self.value_network.parameters(),
            lr=self.hparams.value_lr,
            eps=1e-5
        )
        return value_optimizer, policy_optimizer

    def train_dataloader(self):
        return DataLoader(dataset=self.dataset, batch_size=self.hparams.batch_size)

    def training_step(self, batch, batch_idx):
        opt_value, opt_policy = self.optimizers()
        state, action, old_log_prob, advantages, returns = batch

        device = self.device
        state = state.to(device)
        action = action.to(device)
        old_log_prob = old_log_prob.to(device)
        advantages = advantages.to(device)
        returns = returns.to(device)


        state_values = self.value_network(state)
        value_loss = F.mse_loss(state_values, returns)

        opt_value.zero_grad()
        self.manual_backward(value_loss)
        nn.utils.clip_grad_norm_(self.value_network.parameters(), self.hparams.max_grad_norm)
        opt_value.step()


        mean, std = self.policy(state)
        dist = Normal(mean, std)
        new_log_prob = dist.log_prob(action).sum(dim=-1, keepdim=True)

        ratio = torch.exp(new_log_prob - old_log_prob)
        surrogate1 = ratio * advantages
        surrogate2 = torch.clamp(ratio, 1 - self.hparams.epsilon, 1 + self.hparams.epsilon) * advantages

        policy_loss = -torch.min(surrogate1, surrogate2).mean()


        entropy = dist.entropy().sum(dim=-1).mean()
        entropy_loss = -self.hparams.entropy_weight * entropy

        total_policy_loss = policy_loss + entropy_loss

        opt_policy.zero_grad()
        self.manual_backward(total_policy_loss)
        nn.utils.clip_grad_norm_(self.policy.parameters(), self.hparams.max_grad_norm)
        opt_policy.step()


        self.log('Policy_loss', policy_loss, prog_bar=True)
        self.log('Value_loss', value_loss, prog_bar=True)
        self.log('Entropy', entropy, prog_bar=True)
        self.log('Approx_KL', (old_log_prob - new_log_prob).mean(), prog_bar=False)

        return None

    def on_train_epoch_end(self):

        if self.current_epoch % 5 == 0:
            avg_return = test_agent(
                self.test_env,
                self.hparams.number,
                self.policy,
                episodes=5
            )
            self.log("Average_return", avg_return, prog_bar=True)


        if self.current_epoch % 20 == 0:
            video = create_video(
                self.test_env,
                self.hparams.number,
                policy=self.policy
            )
            self.videos.append(video)

In [ ]:
algo = PPO("Walker2d-v5")
trainer = Trainer(
    accelerator='gpu' if num_gpus > 0 else 'cpu',
    devices=num_gpus if num_gpus > 0 else 1,
    max_epochs=500,
    callbacks=[checkpoint_callback],
    enable_progress_bar=True
)

trainer.fit(algo)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ policy        │ GradientPolicy │  139 K │ train │     0 │
│ 1 │ value_network │ ValueNetwork   │  137 K │ train │     0 │
└───┴───────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 277 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 277 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Couldnt run model entirely due to GPU limits and hence couldnt create videos as well due to runtime timing out. Agent gathered an average reward of 3437.419 as shown

In [ ]:
algo.videos[8]

In [ ]:
algo.videos[-2]

In [ ]:
algo.videos[-1]